# Density-Dependent False Removal Rate in Cross-Channel NN Filtering

**Goal:** Understand how spot density within a cell drives spurious (chance) cross-channel nearest-neighbor matches, and quantify the *false removal rate* — the fraction of real spots that would be flagged as crosstalk purely due to geometric crowding, not biology.

This notebook is a **planning document**. Each section explains the question, the approach, and what code/data would be needed. No analysis code is written yet.

## 1. Problem Statement

In `pooled-nn-analysis.py`, cross-channel NN distances are computed: for every spot in channel A, find its nearest neighbor in channel B. A short distance (< threshold *t*) signals potential crosstalk/bleed-through.

But even for **randomly placed spots with no crosstalk**, NN distances shrink as density rises. At high enough density:
- Almost every spot will have a cross-channel neighbor within *t* µm by pure chance.
- The shuffled null distribution (used in `shuffle_spots_within_cells`) already demonstrates this — the shuffled control captures within-cell geometry, but doesn't give a clean analytical handle.

**Key questions:**
1. At what spot density does chance proximity dominate at a given threshold *t*?
2. Where do real cells fall on that density axis?
3. How does the threshold *t* need to be adjusted (or the removal strategy changed) at high densities?

## 2. Theoretical Background

### 2a. Poisson void probability (3D)

Model: $N_B$ spots in channel B drawn uniformly at random from a cell of volume $V_{cell}$. The number density is $\rho_B = N_B / V_{cell}$ (spots/µm³).

For a query spot in channel A, the probability that *no* channel-B spot falls within a sphere of radius *t* is the Poisson void probability:

$$P(\text{NN}_B > t) = e^{-\rho_B \cdot V_{sphere}(t)} = e^{-\rho_B \cdot \frac{4}{3}\pi t^3}$$

So the **false removal rate** (probability that the nearest channel-B neighbor is closer than *t* by chance) is:

$$\text{FR}(\rho_B,\, t) = 1 - e^{-\rho_B \cdot \frac{4}{3}\pi t^3}$$

This is exact in the infinite-volume limit (rare enough spots that boundary effects and self-avoidance are negligible).

### 2b. Anisotropic voxels

Real voxel size is **x=y=0.24 µm, z=1.0 µm**. The optical PSF is elongated along Z. The effective search volume around a spot is not a sphere — it's an ellipsoid:

$$V_{ellipsoid}(t_{xy},\, t_z) = \frac{4}{3}\pi\, t_{xy}^2\, t_z$$

where $t_{xy} = t$ in the XY plane and $t_z = t \cdot (z_{\text{voxel}} / xy_{\text{voxel}})$ if *t* is specified in µm. For an isotropic 3D threshold in µm the sphere formula still applies (distances are already in µm), but the effective number of voxels searched is much smaller in Z due to the coarser sampling — meaning Z resolution is effectively limited. This needs to be kept in mind when interpreting 3D Euclidean thresholds.

### 2c. Expected density in real cells

A rough estimate to sanity-check the theory:
- Typical cortical neuron soma diameter: ~15 µm → $V_{cell} \approx 1770$ µm³
- If one channel has 50 spots in that cell: $\rho_B = 50/1770 \approx 0.028$ spots/µm³
- At *t* = 1 µm: FR = $1 - e^{-0.028 \cdot 4.19} \approx 11\%$
- At *t* = 0.5 µm: FR = $1 - e^{-0.028 \cdot 0.52} \approx 1.5\%$
- If that same cell has 500 spots (highe gene expression): $\rho_B = 0.28$ spots/µm³
- At *t* = 1 µm: FR ≈ 69%  ← most spots removed by chance
- At *t* = 0.5 µm: FR ≈ 13%

## 3. Simulation Plan

Even though the analytical formula above is a good approximation, a simulation is valuable because:
- Real cells are not infinite uniform spaces (boundary effects matter at small *N*).
- We can test anisotropy directly using the actual voxel geometry.
- It validates the theory and gives confidence intervals.

### 3a. Simulation setup

For each density condition:
1. **Define a synthetic cell**: e.g., a cube of side $L$ in pixels (≈ cell bounding box). Convert to µm using `VOXEL_SIZE`. Typical range: $L \in [30, 80]$ pixels in XY → 7.2–19.2 µm, and $L_z \in [10, 30]$ z-slices → 10–30 µm.
2. **Place spots**: Draw $N_A$ and $N_B$ positions uniformly at random within the cell volume (in pixel space, then convert to µm for distance calculations). Use `np.random.default_rng` for reproducibility.
3. **Compute cross-channel NN**: Use `scipy.spatial.KDTree` on µm coordinates, query channel-A spots against channel-B spots.
4. **Apply threshold**: Flag spots where NN distance < *t* µm. Record the fraction flagged.
5. **Repeat** across a sweep of $N_B$ values (and thus $\rho_B$) with many replicates per condition to get variance.

### 3b. Parameters to sweep

| Parameter | Values |
|-----------|--------|
| $N_B$ (spots per channel) | 5, 10, 20, 50, 100, 200, 500 |
| Cell volume ($L^3$ in µm³) | ~500, 1000, 2000, 5000 µm³ |
| Threshold *t* | 0.5, 1.0, 1.5, 2.0 µm |
| Replicates per condition | 200 |

Density axis: $\rho_B = N_B / V_{cell}$, sweep from ~0.001 to ~1.0 spots/µm³.

### 3c. Quantities to record per replicate

- `frac_removed`: fraction of A-spots with NN_B < *t* (the false removal rate under the null)
- `median_nn_dist`: median NN distance in µm
- `n_a`, `n_b`, `rho_b`, `V_cell`, `threshold`

## 4. Plotting Plan

### Figure 1: False removal rate vs density (main result)

- **X-axis**: $\rho_B$ (spots/µm³), log scale from ~0.001 to ~1.0
- **Y-axis**: false removal rate (fraction of spots flagged by chance), 0–1
- **Lines**: one per threshold *t* (0.5, 1.0, 1.5, 2.0 µm) — both the analytic curve and simulation mean ± std
- **Vertical bands**: overlay the density range observed in real data (see Section 5)
- **Reference lines**: FR = 0.05 and FR = 0.10 to indicate 5%/10% false-removal levels

The goal is to read off: *"if my threshold is 1 µm, I start getting >5% false removals once density exceeds X spots/µm³"*.

### Figure 2: Expected NN distance distribution vs density

- For several representative $\rho_B$ values, plot the full NN distance PDF (from simulations)
- Overlay the observed shuffled distribution from real data (the `dists_shuf` output from `pooled-nn-analysis.py`)
- This shows whether real shuffled data matches the uniform random model — if it doesn't, the cells have non-uniform spot distributions that need a different null model

### Figure 3: Safe threshold as a function of density

- **X-axis**: $\rho_B$ (spots/µm³)
- **Y-axis**: the threshold *t* at which FR = 5% (solve: $t^* = \left(-\frac{3\ln(0.95)}{4\pi\rho_B}\right)^{1/3}$)
- This directly tells you: *"at this cell's density, you should not use a threshold tighter than t*"*
- Overlay real cells as scatter points (density on X, currently-used threshold on a horizontal reference line)

## 5. Grounding in Real Data

To overlay real cell densities on the theoretical curves, we need two quantities per cell:
1. **Cell volume** in µm³
2. **Spot count per channel** per cell

### 5a. What data we have

- `pw_ds.load_aggregated_cxg()` → cell × gene table with spot counts per cell per channel. This gives $N_B$ per cell directly.
- Cell volume is NOT currently in the pipeline outputs. Options:
  - **Bounding box proxy**: use per-cell x/y/z span of spots as a rough cell size. This underestimates volume but is available without extra data loading.
  - **Segmentation mask volume**: if cell mask files are accessible, compute exact volume. This is the best option but requires access to mask data.
  - **Fixed assumed volume**: use a literature estimate (e.g., 1500 µm³ for cortical neuron, 500 µm³ for small interneuron) as sensitivity analysis.

### 5b. What to compute

For each cell in each round:
- Estimate $\rho_B = N_B / V_{cell}$ for each channel B
- This gives a cloud of (density, channel) points
- Summarize as percentile range (e.g., 25th–75th) to form the overlay bands on Figure 1

### 5c. Which mice / rounds to pull from

Start with mouse `767022` R2 (the one used in CROSSTALK_TARGETS). Use `select_cells_for_analysis` with `percentile=0` to get all cells, not just high-signal ones — density analysis needs the full range.

### 5d. Channels to focus on

Adjacent pairs from `CHAN_ORDER = ["488", "514", "561", "594", "638"]` — particularly `514→488` and `561→594` where crosstalk is most expected. Use the **mixed** (pre-unmixed) spots for the worst-case density estimate.

## 6. Implications for Removal Strategy

Once the density-FR curve is established, a few strategies emerge:

| Strategy | Description |
|----------|-------------|
| **Fixed threshold with density gate** | Only apply NN removal to cells below a density ceiling where FR < 5%; skip removal for dense cells. |
| **Density-adaptive threshold** | Use $t^*(\rho_B)$ from Figure 3 — tighten threshold for dense cells. |
| **Rate-based removal** | Instead of a hard threshold, flag the top-K% of NN distances; K chosen to match expected crosstalk rate from the unmixing matrix (not purely density). |
| **Two-sample test per cell** | Compare observed adjacent-channel NN distribution to the shuffled null within that cell; only remove spots where there is a statistically significant excess at short distances. |

The current approach (fixed threshold applied uniformly) can be expected to **over-remove** in dense cells and **under-remove** in sparse cells. The goal of this analysis is to characterize how large that effect is for our actual data.

## 7. Implementation Order

1. **Analytic curve** — implement `false_removal_rate(rho, t)` and `safe_threshold(rho, alpha=0.05)` using the Poisson formula. Plot Figure 1 and Figure 3 analytically. (~30 min, no data needed)
2. **Simulation** — implement the Monte Carlo simulation for a single cell volume, sweep over $N_B$, confirm it matches the analytic curve. (~1 hr)
3. **Real density extraction** — load `pw_ds` for mouse `767022` R2, extract per-cell spot counts and bounding-box volumes; compute $\rho_B$ for each channel. (~1 hr)
4. **Overlay** — combine analytic curves + simulation + real data density bands into the final 3-panel figure. (~1 hr)
5. **Interpretation** — annotate the currently-used threshold(s) on the plot and discuss recommended adjustment.